# ARC3 observation-driven exploration candidate

HDS候補照会と独立した出所を持ち、公開盤面と直前の公開応答から合法Actionを一手ずつ選ぶ候補です。HDSのHOLDはActionの認可へ変換しません。Save & Run Allはdummy parquetを作るだけで、Competition Rerunの結果ではありません。HDS/接続ソースのみを埋め込み、入力・事前記憶・評価記録・認証情報は含めません。

In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
import base64, hashlib, io, json, sys, zipfile
from pathlib import Path, PurePosixPath

_BUNDLE_SHA256 = "e2a9cbe50247c63caf46a5395f5524e573554cd46be5cabc220518c65f8d3627"
_BUNDLE_B64 = """..."""
payload = base64.b64decode(_BUNDLE_B64, validate=True)
if hashlib.sha256(payload).hexdigest() != _BUNDLE_SHA256:
    raise RuntimeError("ARC3 source bundle SHA256 mismatch")
root = Path("/tmp/arc3-hds-source")
root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    if archive.testzip() is not None:
        raise RuntimeError("ARC3 source bundle ZIP is corrupt")
    names = archive.namelist()
    for name in names:
        path = Path(name)
        if path.is_absolute() or ".." in path.parts or not (
            name.startswith("接続/") or name.startswith("HDS/学習系統/v0.4.2/hds学習系統/")
            or name == "ARC3-SOURCE-MANIFEST.json"
        ):
            raise RuntimeError(f"Unexpected source bundle path: {name}")
    archive.extractall(root)
manifest = json.loads((root / "ARC3-SOURCE-MANIFEST.json").read_text(encoding="utf-8"))
for item in manifest["含有"]:
    path = root / PurePosixPath(item["パス"])
    raw = path.read_bytes()
    if len(raw) != item["bytes"] or hashlib.sha256(raw).hexdigest() != item["sha256"]:
        raise RuntimeError(f"Source file verification failed: {item['パス']}")
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "HDS/学習系統/v0.4.2"))
import compileall
if not compileall.compile_dir(str(root / "接続"), quiet=1, force=True):
    raise RuntimeError("Connection source compile failed")
if not compileall.compile_dir(str(root / "HDS/学習系統/v0.4.2/hds学習系統"), quiet=1, force=True):
    raise RuntimeError("HDS source compile failed")
from hds学習系統 import HDS学習系統
from 接続.ARC3境界 import 観測する
from 接続.共通観測経路 import 公開観測から候補を照会する
print("Verified source files:", len(manifest["含有"]))
print("Verified HDS:", HDS学習系統.__module__)
    

In [ ]:
%%writefile /tmp/my_agent.py
"""HDS-HOLDと分離した公開観測ベースのARC3探索agent。"""
from __future__ import annotations

import logging
from typing import Any

from agents.agent import Agent
from arcengine import FrameData, GameAction, GameState

from 接続.ARC3境界 import 境界違反, 観測, 観測する
from 接続.共通観測経路 import 公開観測から候補を照会する
from 接続.観測駆動探索Action import 探索Action, 観測駆動探索方針
from hds学習系統 import HDS学習系統

_LOG = logging.getLogger(__name__)


class MyAgent(Agent):
    """探索Actionの出所をHDS候補・確定Actionから分離して実行する。"""

    # Agent.main uses <=, so 79 permits at most 80 actual action requests.
    MAX_ACTIONS = 79

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self._hds = HDS学習系統()
        self._policy = 観測駆動探索方針()
        self._last_hold: dict[str, Any] | None = None
        self._pending: tuple[観測, 探索Action] | None = None
        self._observation_counter = 0
        self._used_observations: set[str] = set()
        self._safe_stop = False

    @property
    def last_hold(self) -> dict[str, Any] | None:
        """試験・診断向けのHDS要約。盤面や候補内容は保持しない。"""
        return None if self._last_hold is None else dict(self._last_hold)

    @property
    def exploration_stats(self) -> dict[int, dict[str, float | int]]:
        return self._policy.統計()

    def _現在のSDK観測(self, fallback: FrameData) -> FrameData | None:
        if self.arc_env is None:
            return fallback
        raw = getattr(self.arc_env, "observation_space", None)
        if raw is None:
            return None
        return self._convert_raw_frame_data(raw)

    def _HDS照会後に停止しない(self, frames: list[FrameData], frame: FrameData) -> None:
        """HDS HOLDを独立診断として記録する。探索Actionの認可には使わない。"""
        try:
            observation = 観測する(frame, 観測通番=max(0, len(frames) - 1))
            candidate = 公開観測から候補を照会する(self._hds, observation)
            self._last_hold = {
                "状態": "HOLD",
                "観測指紋": observation.指紋,
                "候補経路状態": candidate.get("状態"),
                "次作用候補数": len(candidate.get("次作用候補群", [])),
                "候補側申告実行許可": candidate.get("実行許可") is True,
                "探索Actionとの関係": "独立・認可に不使用",
            }
            _LOG.info(
                "ARC3 HDS diagnostic HOLD: candidate_state=%s candidate_count=%d",
                self._last_hold["候補経路状態"],
                self._last_hold["次作用候補数"],
            )
        except Exception as exc:
            self._last_hold = {
                "状態": "HOLD",
                "理由": type(exc).__name__,
                "候補側申告実行許可": False,
                "探索Actionとの関係": "独立・認可に不使用",
            }
            _LOG.warning("ARC3 HDS diagnostic HOLD (%s)", type(exc).__name__)

    @staticmethod
    def _状態(frame: FrameData) -> str:
        value = frame.state
        return getattr(value, "name", str(value))

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        if self._safe_stop or self.action_counter > self.MAX_ACTIONS:
            return True
        state = self._状態(latest_frame)
        if state in ("WIN", "GAME_OVER"):
            return True

        if state == "NOT_PLAYED" and self.action_counter == 0:
            current = self._現在のSDK観測(latest_frame)
            if current is None:
                self._safe_stop = True
                return True
            current_state = self._状態(current)
            if current_state in ("WIN", "GAME_OVER"):
                return True
            if current_state not in ("NOT_PLAYED", "NOT_FINISHED"):
                self._safe_stop = True
                return True
            if current_state == "NOT_FINISHED":
                try:
                    observed = 観測する(current, 観測通番=self._observation_counter)
                except 境界違反 as exc:
                    self._last_hold = {"状態": "HOLD", "理由": type(exc).__name__}
                    self._safe_stop = True
                    return True
                if not observed.作用可能番号:
                    return True
                if self._last_hold is None:
                    self._HDS照会後に停止しない(frames, current)
                return False
            return False

        if state == "NOT_FINISHED":
            try:
                observed = 観測する(latest_frame, 観測通番=self._observation_counter)
            except 境界違反 as exc:
                self._last_hold = {"状態": "HOLD", "理由": type(exc).__name__}
                self._safe_stop = True
                return True
            if not observed.作用可能番号:
                return True
            if self._last_hold is None:
                self._HDS照会後に停止しない(frames, latest_frame)
            return False

        # A reset is allowed only for the initial NOT_PLAYED state.
        return True

    def choose_action(
        self, frames: list[FrameData], latest_frame: FrameData
    ) -> 探索Action:
        if self._safe_stop:
            raise 境界違反("安全停止中はActionを選べない")
        if self._pending is not None:
            raise 境界違反("前Actionの応答観測前に次Actionを選べない")
        observed = 観測する(latest_frame, 観測通番=self._observation_counter)
        if observed.指紋 in self._used_observations:
            raise 境界違反("同一観測に二度作用できない")
        action = self._policy.選ぶ(observed)
        if action is None:
            raise 境界違反("公開観測から合法な探索Actionを構成できない")
        if action.作用番号 == 0 and not (
            observed.状態 == "NOT_PLAYED" and self.action_counter == 0
        ):
            raise 境界違反("RESETは初回NOT_PLAYED状態でのみ許可")
        self._pending = (observed, action)
        return action

    def do_action_request(self, action: 探索Action) -> FrameData:
        """最新のSDK観測・合法集合・座標を再照合して一回だけ送信する。"""
        if not isinstance(action, 探索Action) or self._pending is None:
            raise 境界違反("探索出所のないActionをSDKへ送れない")
        before, pending_action = self._pending
        if action != pending_action or action.出所 != "観測駆動探索":
            raise 境界違反("送信Actionが現在観測の探索判断と一致しない")
        if before.指紋 in self._used_observations:
            raise 境界違反("同一観測に二度作用できない")
        if self.arc_env is None:
            raise 境界違反("SDK環境がないため探索Actionを送れない")

        raw_current = getattr(self.arc_env, "observation_space", None)
        if raw_current is None:
            raise 境界違反("作用前のSDK観測がない")
        current_frame = self._convert_raw_frame_data(raw_current)
        current = 観測する(current_frame, 観測通番=before.観測通番)
        if current.指紋 != before.指紋:
            raise 境界違反("作用前にSDK観測が変化した")
        if action.作用番号 != 0 and action.作用番号 not in current.作用可能番号:
            raise 境界違反("探索Actionが最新available_actionsにない")
        if action.作用番号 == 0 and current.状態 != "NOT_PLAYED":
            raise 境界違反("RESETは初回NOT_PLAYED状態でのみ許可")
        if action.作用番号 == 6:
            if set(action.引数) != {"x", "y"}:
                raise 境界違反("座標Actionにx,yが必要")
            for key in ("x", "y"):
                value = action.引数[key]
                if type(value) is not int or not 0 <= value <= 63:
                    raise 境界違反("座標Actionが公式範囲外")
        elif action.引数:
            raise 境界違反("単純Actionに引数を付けられない")

        self._used_observations.add(before.指紋)
        raw_after = self.arc_env.step(
            GameAction.from_id(action.作用番号),
            data=dict(action.引数),
            reasoning={"探索出所": action.出所, "作用前観測指紋": before.指紋},
        )
        if raw_after is None:
            self._safe_stop = True
            raise 境界違反("SDKから公開応答を取得できない")
        frame = FrameData.model_validate(self._convert_raw_frame_data(raw_after))
        after = 観測する(frame, 観測通番=before.観測通番 + 1)
        if action.作用番号 != 0:
            receipt = self._policy.応答を記録する(before, action, after)
            _LOG.debug("ARC3 exploratory response: %s", receipt)
        self._observation_counter = before.観測通番 + 1
        self._pending = None
        return frame


In [ ]:
import os
from pathlib import Path
import shutil
import subprocess

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    subprocess.run(
        ["curl", "--fail", "--retry", "999", "--retry-all-errors",
         "--retry-delay", "5", "--retry-max-time", "600",
         "http://gateway:8001/api/games"],
        check=True,
    )
    framework = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents")
    working = Path("/kaggle/working/ARC-AGI-3-Agents")
    shutil.copytree(framework, working, dirs_exist_ok=True)
    shutil.copyfile(
        "/tmp/my_agent.py",
        working / "agents/templates/my_agent.py",
    )
    (working / "agents/__init__.py").write_text(
        """from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent
load_dotenv()
AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}
""",
        encoding="utf-8",
    )
    (working / ".env").write_text(
        """SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/tmp/server_recording
""",
        encoding="utf-8",
    )
    env = os.environ.copy()
    env["PYTHONPATH"] = "/tmp/arc3-hds-source:/tmp/arc3-hds-source/HDS/学習系統/v0.4.2"
    env["MPLBACKEND"] = "agg"
    subprocess.run(["python", "main.py", "--agent", "myagent"], cwd=working, env=env, check=True)
else:
    # Save & Run Allのdummy parquet。Competition Rerunの得点や受理を表すものではない。
    import pandas as pd
    submission = pd.DataFrame(
        data=[["1_0", "1", True, 1]],
        columns=["row_id", "game_id", "end_of_game", "score"],
    )
    submission.to_parquet("/kaggle/working/submission.parquet", index=False)
    print("Save & Run All placeholder created; no competition score was evaluated.")
